In [5]:
!pip install -q google-genai pandas matplotlib

In [6]:
from google import genai

import pandas as pd
import sqlite3
import re

from datetime import date
from datetime import datetime
from datetime import timedelta

In [8]:
client = genai.Client(
    api_key="enter_API_key_here"
)

MODEL_NAME = "gemini-2.5-flash"

print("🌸 Maya is awake!")

🌸 Maya is awake!


In [10]:
df = pd.read_csv("FedCycleData071012 (2).csv")

print(df.head())

print("\nRows:", len(df))

  ClientID  CycleNumber  Group  CycleWithPeakorNot  ReproductiveCategory  \
0  nfp8122            1      0                   1                     0   
1  nfp8122            2      0                   1                     0   
2  nfp8122            3      0                   1                     0   
3  nfp8122            4      0                   1                     0   
4  nfp8122            5      0                   1                     0   

   LengthofCycle MeanCycleLength EstimatedDayofOvulation LengthofLutealPhase  \
0             29           27.33                      17                  12   
1             27                                      15                  12   
2             29                                      15                  14   
3             27                                      15                  12   
4             28                                      16                  12   

  FirstDayofHigh  ... Method Prevmethod Methoddate Whychart Ne

In [13]:
conn = sqlite3.connect("maya_memory.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT,
    symptom TEXT,
    severity INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS periods (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    start_date TEXT
)
""")

conn.commit()

print("🌸 Database ready.")

🌸 Database ready.


In [14]:
SYMPTOMS = [
    "cramps",
    "nausea",
    "bloating",
    "headache",
    "fatigue",
    "tired",
    "back pain",
    "mood swings",
    "dizziness",
    "vomiting",
    "anxiety",
    "acne",
    "breast tenderness"
]

In [15]:
def save_symptom(date, symptom, severity):

    cursor.execute(
        """
        INSERT INTO logs
        (date, symptom, severity)
        VALUES (?, ?, ?)
        """,
        (date, symptom, severity)
    )

    conn.commit()


def extract_data(text):

    text = text.lower()

    found = []

    for symptom in SYMPTOMS:
        if symptom in text:
            found.append(symptom)

    severity = None

    match = re.search(
        r'(?:pain|severity|level)\s*(?:is)?\s*(\d+)',
        text
    )

    if match:
        severity = int(match.group(1))

    return found, severity


def get_logs():

    cursor.execute("""
    SELECT date, symptom, severity
    FROM logs
    ORDER BY id DESC
    LIMIT 5
    """)

    return cursor.fetchall()


def show_logs():

    cursor.execute("""
    SELECT date, symptom, severity
    FROM logs
    ORDER BY date DESC
    """)

    return cursor.fetchall()

In [16]:
def save_period(period_date):

    cursor.execute(
        "INSERT INTO periods (start_date) VALUES (?)",
        (period_date,)
    )

    conn.commit()


def average_cycle():

    cursor.execute("""
    SELECT start_date
    FROM periods
    ORDER BY start_date
    """)

    dates = cursor.fetchall()

    if len(dates) < 2:
        return None

    lengths = []

    for i in range(1, len(dates)):

        d1 = datetime.strptime(
            dates[i-1][0],
            "%Y-%m-%d"
        )

        d2 = datetime.strptime(
            dates[i][0],
            "%Y-%m-%d"
        )

        lengths.append((d2 - d1).days)

    return sum(lengths) / len(lengths)


def predict_next_period():

    cursor.execute("""
    SELECT start_date
    FROM periods
    ORDER BY start_date DESC
    LIMIT 1
    """)

    latest = cursor.fetchone()

    avg = average_cycle()

    if latest is None or avg is None:
        return "Not enough cycle data."

    last_date = datetime.strptime(
        latest[0],
        "%Y-%m-%d"
    )

    prediction = last_date + timedelta(
        days=round(avg)
    )

    return prediction.strftime("%Y-%m-%d")

In [17]:
def average_pain():

    cursor.execute("""
    SELECT AVG(severity)
    FROM logs
    """)

    value = cursor.fetchone()[0]

    if value is None:
        return 0

    return round(value, 1)


def common_symptoms():

    cursor.execute("""
    SELECT symptom, COUNT(*)
    FROM logs
    GROUP BY symptom
    ORDER BY COUNT(*) DESC
    """)

    return cursor.fetchall()

In [18]:
def pain_insight():

    avg = average_pain()

    if avg >= 8:
        return "Pain levels have been high."

    elif avg >= 5:
        return "Pain levels are moderate."

    return "Pain levels are generally mild."


def symptom_insight():

    symptoms = common_symptoms()

    if symptoms:
        return f"Most common symptom: {symptoms[0][0]}."

    return "No symptom patterns yet."


def cycle_insight():

    avg = average_cycle()

    if avg is None:
        return "Not enough cycle data."

    return f"Average cycle length: {avg:.1f} days."


def get_insights():

    return [
        pain_insight(),
        symptom_insight(),
        cycle_insight()
    ]

In [19]:
conversation_memory = []

EMOTIONS = {
    "stressed": "stress",
    "anxious": "anxiety",
    "worried": "anxiety",
    "sad": "sadness",
    "tired": "fatigue",
    "exhausted": "fatigue",
    "angry": "irritability"
}


def detect_emotion(text):

    text = text.lower()

    for word, emotion in EMOTIONS.items():
        if word in text:
            return emotion

    return None


def remember(text):

    conversation_memory.append(text)

    if len(conversation_memory) > 10:
        conversation_memory.pop(0)

In [22]:
print("🌸 Maya Ultimate is ready!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("🌸 Maya: Take care! ❤️")
        break

    remember(user_input)

    # PERIOD LOGGING

    if (
        "period started" in user_input.lower()
        or "got my period" in user_input.lower()
    ):

        today = str(date.today())

        save_period(today)

        print(
            f"\n🌸 Maya: I've logged {today} as the start of your period.\n"
        )

        continue

    # PREDICTION

    if (
        "predict" in user_input.lower()
        or "next period" in user_input.lower()
    ):

        prediction = predict_next_period()

        print(
            f"\n🌸 Maya: Your next period may begin around {prediction}.\n"
        )

        continue

    # INSIGHTS

    if "insight" in user_input.lower():

        insights = get_insights()

        print("\n🌸 Maya's Insights:\n")

        for insight in insights:
            print("•", insight)

        print()

        continue

    # LOGS

    if "show logs" in user_input.lower():

        logs = show_logs()

        print()

        for log in logs:
            print(
                f"{log[0]} | {log[1]} | {log[2]}/10"
            )

        print()

        continue

    # SYMPTOMS

    symptoms, severity = extract_data(user_input)

    if symptoms:

        today = str(date.today())

        for symptom in symptoms:

            save_symptom(
                today,
                symptom,
                severity if severity else 5
            )

    emotion = detect_emotion(user_input)

    memory = get_logs()

    prompt = f"""
You are Maya, an AI menstrual health companion.

Recent symptom logs:
{memory}

Conversation history:
{conversation_memory}

Detected emotion:
{emotion}

Provide supportive menstrual health guidance.

Never claim to be a doctor.

User:
{user_input}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    print("\n🌸 Maya:")
    print(response.text)
    print()

🌸 Maya Ultimate is ready!
Type 'exit' to stop.

You: I'm stressed and my cramps are terrible today.

🌸 Maya:
Oh, I'm so sorry to hear you're feeling stressed and that your cramps are terrible today. That sounds incredibly tough to deal with, especially with everything else going on.

It's completely understandable to feel extra sensitive or overwhelmed during your period, and stress can definitely make symptoms like cramps feel even more intense.

Here are a few things that might help you find some comfort:

*   **For your cramps:**
    *   **Heat:** A warm bath or shower, or a heating pad/hot water bottle on your lower abdomen can be wonderfully soothing.
    *   **Gentle movement:** If you feel up to it, a short, gentle walk or some light stretching can sometimes help relax your muscles.
    *   **Hydration:** Make sure you're drinking plenty of water, as dehydration can sometimes make cramps worse.

*   **For your stress:**
    *   **Deep breathing:** Taking a few slow, deep breaths